# 05b Contrast sweep

Reruns the noise-free matrix imaging of `05_matrix_imaging` for one or two scenarios with a Vp drop of 1, 2.5, 5
and 10 % inside the BDT, and finds the smallest drop the best design can still detect.

**Read this before trusting the answer.** In 05 a column counts as imaged when its peak is `SNR_MIN` times the
image's *own* median. With no noise the Born image scales almost exactly linearly with the drop, so that relative
test gives the same coverage at every drop: the noise-free 05 score cannot tell 1 % from 10 %. This notebook
therefore scores each image twice:

- **relative**: 05's own score, unchanged (expect it to be flat across drops);
- **absolute**: the same score, but the detection floor is held fixed at the background (median) of the
  `DROP_REF` image. This stands in for a fixed noise level until `06_noise` exists, and the answer depends on it.

Inputs: everything 05 reads, plus `05_matrix_imaging.ipynb` itself (its code is reused, not copied).
Outputs: `data/results_05b_contrast.csv`, images in `data/mi_images_drop*/`

In [ ]:
# --- Imports ---
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# --- Settings ---
DROPS = [0.01, 0.025, 0.05, 0.10]           # fractional Vp drop inside the BDT
SWEEP_SCENARIOS = ["pos00_dome", "pos09_steeper"]   # one centred, one off-centre and narrow
BEST = None                                 # design to test; None = highest mean coverage in results_05_mi.csv
DROP_REF = 0.05                             # its image background sets the fixed (absolute) detection floor
DETECT_COV = 0.5                            # detected = at least this fraction of the BDT top imaged within TOP_TOL
DETECT_CREST = 1.0                          # ... and the BDT high picked within this many km
OUT = "../data/results_05b_contrast.csv"

In [ ]:
# --- Load 05's code: settings, inputs and functions (everything before its run cell) ---
src = "".join("".join(c["source"]) + "\n" for c in json.load(open("05_matrix_imaging.ipynb"))["cells"]
              if c["cell_type"] == "code")
cells = src.split("# %%")
RUN = next(i for i, c in enumerate(cells) if "# --- Run:" in c)
ns = {}
exec("# %%".join(cells[:RUN]), ns)
run_cell = cells[RUN]

if BEST is None:
    prev = pd.read_csv(f"{ns['DATA']}/results_05_mi.csv")
    BEST = prev.groupby("design").coverage.mean().idxmax()
print("best design:", BEST)

In [ ]:
# --- Sweep: run 05's run cell once per drop, for the best design only ---
for drop in DROPS:
    ns.update(DROP=drop, SCENARIOS=SWEEP_SCENARIOS, DESIGNS=[BEST],
              RESULTS=f"{ns['DATA']}/results_05_mi_drop{drop:g}.csv",
              IMAGE_DIR=f"{ns['DATA']}/mi_images_drop{drop:g}")
    print(f"--- drop {drop:.1%}")
    exec(run_cell, ns)

In [ ]:
# --- Score each image against 05's relative floor and against a fixed floor from the DROP_REF image ---
pick_top, crest, SC, sc_npz = ns["pick_top"], ns["crest"], ns["SC"], ns["sc_npz"]
IX, IY, mx, my, ii, ij = (ns[k] for k in ("IX", "IY", "mx", "my", "ii", "ij"))
key = BEST.replace(" ", "_")


def load(drop, name):
    return np.load(f"{ns['DATA']}/mi_images_drop{drop:g}/{name}.npz")[key]


def score_floor(img, name, floor):
    """05's score, but a column is imaged only if its peak is SNR_MIN x floor (floor fixed across drops)."""
    centre = np.array(SC["scenarios"][name]["centre"])
    true = sc_npz[f"top_{name}"][np.ix_(mx[ii], my[ij])]
    est = pick_top(img, ns["SNR_MIN"] * floor)
    XX, YY = np.meshgrid(IX, IY, indexing="ij")
    target = (np.hypot(XX - centre[0], YY - centre[1]) <= ns["TARGET_R"]) & np.isfinite(true)
    good = target & (np.abs(est - true) <= ns["TOP_TOL"])
    strong = pick_top(img, max(ns["CREST_FRAC"] * img.max(), ns["SNR_MIN"] * floor))
    crest_err = (float(np.hypot(*(crest(strong, XX, YY) - crest(true, XX, YY))))
                 if np.isfinite(strong).any() else np.nan)
    return good.sum() / max(target.sum(), 1), crest_err


rows = []
for name in SWEEP_SCENARIOS:
    floor = float(np.median(load(DROP_REF, name)))
    for drop in DROPS:
        img = load(drop, name)
        rel = pd.read_csv(f"{ns['DATA']}/results_05_mi_drop{drop:g}.csv").query("scenario == @name and design == @BEST").iloc[0]
        cov_abs, crest_abs = score_floor(img, name, floor)
        rows.append(dict(scenario=name, design=BEST, drop=drop,
                         peak_over_floor=float(img.max() / floor),
                         coverage_rel=rel.coverage, crest_err_rel=rel.crest_err_km,
                         coverage_abs=cov_abs, crest_err_abs=crest_abs,
                         detected=bool(cov_abs >= DETECT_COV and crest_abs <= DETECT_CREST)))
res = pd.DataFrame(rows)
res.to_csv(OUT, index=False)
res.round(3)

In [ ]:
# --- Smallest drop the best design still detects, per scenario ---
for name, g in res.groupby("scenario"):
    ok = g.loc[g.detected, "drop"]
    print(f"{name}: smallest detected drop = {ok.min():.1%}" if len(ok) else f"{name}: not detected at any drop")
flat = res.groupby("scenario").coverage_rel.agg(np.ptp).max()
print(f"relative (05) coverage varies by at most {flat:.3f} across drops"
      + (": as expected, the noise-free relative score is blind to contrast" if flat < 0.02 else ""))

In [ ]:
# --- Coverage against drop ---
with plt.style.context("default"):
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
    for ax, col, lab in zip(axs, ["coverage_rel", "coverage_abs"],
                            ["Relative floor (05 as is)", f"Fixed floor (background at {DROP_REF:.0%})"]):
        for name, g in res.groupby("scenario"):
            ax.plot(g["drop"] * 100, g[col], "o-", label=name)
        ax.axhline(DETECT_COV, color="grey", ls="--", lw=0.8)
        ax.set(xscale="log", xticks=[1, 2.5, 5, 10], xticklabels=["1", "2.5", "5", "10"],
               xlabel="Vp drop (%)", title=lab, ylim=(0, 1.02))
    axs[0].set_ylabel(f"Coverage of BDT top ({BEST})")
    axs[1].legend(frameon=False)
    plt.tight_layout()
    plt.show()